# Comparación de clasificadores — Detector de fallas de GPU

Modelos a comparar, todos sobre el mismo split de episodios (train/test)
que usa `src/train.py` (`random_state=42`):

- **Random Forest** (el elegido en `train.py`)
- **Gradient Boosting**
- **Logistic Regression**
- **SVM (kernel RBF)**
- **KNN (k=5)**

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent  # notebooks/ -> raíz del repo
sys.path.insert(0, str(REPO / "src"))

import features as feat
from schema import validar_y_limpiar_telemetria
from train import separar_episodios_train_test, construir_dataset, RANDOM_STATE

In [2]:
df = pd.read_csv(REPO / "data" / "telemetria_publica.csv")
df = validar_y_limpiar_telemetria(df)

ids_train, ids_test = separar_episodios_train_test(df, test_size=0.25, random_state=RANDOM_STATE)
df_train = df[df["episodio_id"].isin(ids_train)]
df_test = df[df["episodio_id"].isin(ids_test)]

X_train, y_train, _ = construir_dataset(df_train, feat.TAM_VENTANA_DEFAULT)
X_test, y_test, _ = construir_dataset(df_test, feat.TAM_VENTANA_DEFAULT)

print(f"Episodios -> train: {len(ids_train)}, test: {len(ids_test)}")
print(f"Ventanas  -> train: {len(X_train)}, test: {len(X_test)}")

[schema] Se encontraron 3 fila(s) que incumplen el contrato de datos; se descartan antes de entrenar:
 index  column                         check  failure_case
 10907 power_w greater_than_or_equal_to(0.0)         -13.5
 11662 power_w greater_than_or_equal_to(0.0)          -3.9
 13358 power_w greater_than_or_equal_to(0.0)         -18.1


Episodios -> train: 36, test: 12
Ventanas  -> train: 358, test: 119


In [3]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

modelos = {
    "RandomForest (el elegido en train.py)": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "SVM (RBF)": SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}

filas = []
for nombre, clf in modelos.items():
    pipe = Pipeline([("escalador", StandardScaler()), ("clasificador", clf)])

    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    t_fit = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = pipe.predict(X_test)
    t_pred = time.perf_counter() - t0

    filas.append({
        "modelo": nombre,
        "accuracy_test": round(accuracy_score(y_test, y_pred), 4),
        "f1_macro_test": round(f1_score(y_test, y_pred, average="macro"), 4),
        "t_fit_s": round(t_fit, 3),
        "t_pred_s (119 ventanas)": round(t_pred, 4),
    })

resultado = pd.DataFrame(filas)
resultado

,modelo,accuracy_test,f1_macro_test,t_fit_s,t_pred_s (119 ventanas)
0,RandomForest (el elegido en train.py),1.0,1.0,0.660,0.0810
1,GradientBoosting,1.0,1.0,1.125,0.0031
2,LogisticRegression,1.0,1.0,0.016,0.0019
3,SVM (RBF),1.0,1.0,0.014,0.0022
4,KNN (k=5),1.0,1.0,0.003,2.1413


## Conclusión

Los cinco modelos empatan en **accuracy = 1.0 y F1 macro = 1.0** sobre los
episodios de test: las clases están tan separadas en el espacio de features
(temp_c para sobrecalentamiento, ecc_errors para degradacion_memoria, std de
clock_mhz para falla_alimentacion) que casi cualquier clasificador razonable
las resuelve.

Random Forest **no ganó** esta comparación — nadie perdió. Se mantiene como
elección de `train.py` por:

- **Robustez esperada ante datos más ruidosos** que los del dataset público
  (el examen oculto del profesor puede no estar tan limpiamente separado).
  Un ensamble de árboles traza fronteras no lineales por umbrales sin
  necesitar diseñar términos de interacción a mano.
- **`feature_importances_` nativo** — se usó directamente para responder,
  con números, cuál feature delata mejor cada falla (ver README).
- **`predict_proba` sin calibración extra**, necesario para el campo
  `confianza` que pide la API.
- No necesita tuning de hiperparámetros para rendir bien con pocos datos.

La diferencia real entre modelos está en velocidad de entrenamiento
(regresión logística y SVM entrenan ~30x más rápido), pero irrelevante para
este dataset de 358 ventanas de entrenamiento.